# Практична робота 1. Поглиблена робота з рядками та регулярними виразами

Навчальний ноутбук для Google Colab на основі файлу «Практична робота 1». **Мета:** послідовно розібрати шість задач заняття й самостійно застосувати `split`, `replace`, `strip`, `translate`, f-рядки та модуль `re`.

**План заняття:** 1) параметри URL; 2) очищення тексту; 3) перевірка числа й звіт; 4) пошук email; 5) форматування телефону; 6) поділ за кількома роздільниками; 7) підсумкове завдання.

Запускайте комірки згори донизу кнопкою ▶ або клавішами **Shift+Enter**. Усі приклади працюють на стандартному Python, без встановлення пакетів та доступу до інтернету. У комірках для самостійної роботи `None` позначає незаповнений результат. Поки ви не допишете код, комірка виведе підказку й не перерве запуск усього ноутбука.

## 0. Коротка розминка: що повертають методи рядків

Рядок у Python незмінний: `replace` і `strip` створюють новий рядок. `split` створює список. Словник зберігає відповідність ключа та значення. Метод можна запускати на літералі або змінній. Нижче виводимо `repr`, щоб бачити пробіли по краях.

In [ ]:
source = "  data+science  "
print("Початковий:", repr(source))
print("Новий:", repr(source.strip().replace("+", " ")))
print("Початковий після виклику:", repr(source))
print("Частини:", source.strip().split("+"))

## 1. Обробка пошукового URL та словник параметрів

У файлі практичної роботи потрібно відділити параметри після `?`, розділити пари за `&`, а кожну пару за `=`; символ `+` у значенні замінити пробілом. Застосуємо `split('?', 1)`, щоб розділити тільки за **першим** знаком питання; `split('=', 1)` дає аналогічний захист для значення з `=`. Пара `key=value` потрапляє до словника як `parameters[key] = value`.

**Припущення навчального алгоритму:** URL містить `?`, ключі не повторюються, пари мають знак `=`. У реальному URL можливі `%XX`, порожні значення й повторювані ключі; для них є `urllib.parse`. Кутові дужки, якими в конспекті обрамлено посилання, не є частиною URL.

In [ ]:
url = "https://search.example/find?q=solar+wind&lang=uk&page=2"
address, query = url.split("?", 1)
parameters = {}
for pair in query.split("&"):
    key, value = pair.split("=", 1)
    parameters[key] = value.replace("+", " ")
print("Адреса:", address)
print("Рядок параметрів:", query)
print("Словник:", parameters)
assert parameters == {"q": "solar wind", "lang": "uk", "page": "2"}

Читаємо вивід: `page` лишився рядком `'2'`, адже `split` не перетворює типи. Якщо потрібне число, виконайте `int(parameters['page'])` після перевірки. Нижче стандартна бібліотека правильно розкодовує також `%20` і зберігає повтори ключа як список.

In [ ]:
from urllib.parse import urlsplit, parse_qs
realistic_url = "https://example.org/search?q=machine+learning&q=Python%203&empty="
parsed = parse_qs(urlsplit(realistic_url).query, keep_blank_values=True)
print(parsed)  # q має два значення, empty має порожнє значення

### Самостійне завдання 1.1. Параметри каталогу

Обробіть `url` методами `split` і `replace`. Запишіть параметри у словник `result`. Виведіть вихідний рядок параметрів та словник. Не використовуйте `urllib.parse` у цій вправі.

**Очікування:** `category` → `'home office'`, `sort` → `'price'`, `limit` → `'10'`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
url = "https://shop.example/items?category=home+office&sort=price&limit=10"
result = None
query = None
# TODO: відділіть query за першою "?", створіть словник і пройдіть пари

if result is None:
    print("Допишіть розбір URL у місці TODO")
else:
    print("Рядок параметрів:", query)
    print("Словник:", result)
    print("Самоперевірка:", result == {"category": "home office", "sort": "price", "limit": "10"})

### Самостійне завдання 1.2. Реальний URL з повторюваним ключем

Використайте `urlsplit` і `parse_qs` для нового URL. Збережіть словник у `result` і поясніть, чому значенням `tag` є список.

**Очікування:** `tag` → `['python', 'data science']`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
from urllib.parse import urlsplit, parse_qs
url = "https://learn.example/search?tag=python&tag=data+science&level=beginner"
result = None
# TODO: отримайте query через urlsplit, перетворіть його через parse_qs

if result is None:
    print("Заповніть result у місці TODO")
else:
    print(result)
    print("Самоперевірка:", result.get("tag") == ["python", "data science"])

## 2. Очищення рядка та `translate`

`strip()` прибирає пробільні символи **лише по краях**. `str.maketrans('aeiou', '12345')` створює відповідність `a→1`, `e→2`, `i→3`, `o→4`, `u→5`. Довжини двох рядків у цій формі `maketrans` мають збігатися. `translate(table)` замінює символи і повертає новий рядок; великі літери залишаться без змін, якщо не додати їх до таблиці.

In [ ]:
raw = "   Data in cloud   "
clean = raw.strip()
table = str.maketrans("aeiou", "12345")
converted = clean.translate(table)
print("Початок:", repr(raw))
print("Очищено:", repr(clean))
print("Перетворено:", repr(converted))
assert converted == "D1t1 3n cl45d"

Зверніть увагу: `strip()` не прибирає пробіл між словами. `translate` працює із символами, а не шукає слова. Для видалення символів можна створити словник, у якому їхнім значенням буде `None`.

In [ ]:
separator_table = str.maketrans({"-": None, " ": None})
print("AB-12 - 34".translate(separator_table))  # AB1234

### Самостійне завдання 2.1. Голосні у фразі

Очистіть рядок, створіть таблицю для **малих** голосних `aeiou` → `12345`, виведіть вихідний, очищений і перетворений рядки. Збережіть кінцевий рядок у `result`.

**Очікування:** `c4d2 r2v32w 3s 5s2f5l`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
text = "   code review is useful   "
result = None
# TODO: clean = ...; table = ...; result = ...

if result is None:
    print("Додайте strip, maketrans і translate")
else:
    print("Було:", repr(text))
    print("Стало:", repr(result))
    print("Самоперевірка:", result == "c4d2 r2v32w 3s 5s2f5l")

### Самостійне завдання 2.2. Очищення коду замовлення

Видаліть пробіли по краях, а потім усі дефіси й внутрішні пробіли за допомогою `translate`. Збережіть у `result`.

**Очікування:** `UA042026`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
code_raw = "  UA- 04- 2026  "
result = None
# TODO: побудуйте таблицю str.maketrans з None для "-" і " "

if result is None:
    print("Заповніть result")
else:
    print(result)
    print("Самоперевірка:", result == "UA042026")

## 3. `isdigit`, перетворення типу та f-рядки

`input()` повертає рядок. У Colab для повторюваного запуску зручніше задавати приклади введення в коді. `isdigit()` показує, чи рядок непорожній і складається із цифрових символів. Для вправи обмежимо введення ASCII-цифрами `0–9`, щоб кожне прийняте значення гарантовано оброблялося `int`. Перевірка `s.isascii() and s.isdigit()` відкидає також знаки `-`, крапку та літери. Вона не приймає від’ємні числа; для них використовуйте `try/except ValueError` навколо `int(s)`.

Формат `:d` задає десятковий запис, `:#x` — шістнадцятковий із `0x`, `:#o` — вісімковий із `0o`, `:#b` — двійковий із `0b`. F-рядок повертає рядок для виводу.

In [ ]:
def show_number_report(user_input):
    if not (user_input.isascii() and user_input.isdigit()):
        return "Помилка: введіть невід’ємне ціле число цифрами 0–9"
    number = int(user_input)
    return (f"dec: {number:d}\n"
            f"hex: {number:#x}\n"
            f"oct: {number:#o}\n"
            f"bin: {number:#b}")

for value in ("42", "-8", "4.2", "abc"):
    print("Введено:", repr(value))
    print(show_number_report(value))
    print("---")
assert "0x2a" in show_number_report("42")

Для `42` маємо `0x2a`, `0o52` та `0b101010`. Функція ізолює перевірку від виводу, тому її легко перевіряти на кількох значеннях. Форматування не змінює значення числа. Ширина й вирівнювання також задаються після двокрапки: `f'{42:>8d}'` створює поле завширшки 8 символів.

### Самостійне завдання 3.1. Звіт для числа 31

Перевірте введення за допомогою `isascii()` та `isdigit()`, перетворіть на `int` і побудуйте 4 рядки звіту. Для вправи встановлено `user_input = '31'`; після перевірки змініть на `'31x'`.

**Очікування:** Для `31`: `0x1f`, `0o37`, `0b11111`; для `31x`: повідомлення про помилку.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
user_input = "31"
result = None
# TODO: за коректного введення створіть рядок з dec, hex, oct, bin
# TODO: для некоректного введення запишіть у result повідомлення про помилку

if result is None:
    print("Реалізуйте перевірку та звіт")
else:
    print(result)
    print("Самоперевірка для 31:", "0x1f" in result and "0o37" in result and "0b11111" in result)

### Самостійне завдання 3.2. Вирівняний вивід

Створіть за допомогою f-рядка один рядок `result`, де назва `Python` займає поле 12 символів зліва, а число `7` — поле 4 символи справа. Вертикальні риски покажуть межі полів.

**Очікування:** `|Python      ||   7|`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
language = "Python"
count = 7
result = None
# TODO: використайте :<12 та :>4, обраміть кожне поле символами |

if result is None:
    print("Заповніть форматований рядок")
else:
    print(result)
    print("Самоперевірка:", result == "|Python      ||   7|")

## 4. Пошук електронних адрес: `re.findall`

`re.findall(pattern, text)` знаходить усі **неперекривні** фрагменти за шаблоном і повертає список. Префікс `r` зберігає зворотні риски шаблону. `\w` означає літеру, цифру або підкреслення (у Unicode також багато нела­тинських літер); `+` — один чи більше разів; `\.` означає буквальну крапку, тоді як просто `.` — майже будь-який символ. Навчальний шаблон нижче обмежує адреси латинськими літерами, цифрами, підкресленням, крапкою, плюсом і дефісом перед `@`, а також піддоменами після нього. Це **виділення кандидатів із навчального тексту**, не повна валідація email і не перевірка існування адреси.

In [ ]:
import re
text = "Пишіть на tutor.one+lab@learn.example.org або lab-team@campus.edu."
pattern = r"[A-Za-z0-9._+\-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+"
addresses = re.findall(pattern, text)
print(addresses)
assert addresses == ["tutor.one+lab@learn.example.org", "lab-team@campus.edu"]

Дужки `(?:...)` групують без захоплення. Якщо використати звичайні `(...)`, `findall` може повертати вміст груп замість повного збігу. Для простої демонстрації механіки можна почати і з шаблону джерела `r'\w+@\w+\.\w+'`, але він пропускає багато типових адрес.

### Самостійне завдання 4.1. Контакти команди

Знайдіть усі адреси у новому тексті. Візьміть навчальний шаблон прикладу або побудуйте власний. Запишіть список у `result`.

**Очікування:** Список із трьох адрес у порядку появи.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
import re
text = "Команда: anna@demo.org, dev.ops@sample.net та team+alerts@demo.org"
result = None
# TODO: pattern = ...; result = re.findall(...)

if result is None:
    print("Створіть шаблон і викличте re.findall")
else:
    print(result)
    print("Самоперевірка:", result == ["anna@demo.org", "dev.ops@sample.net", "team+alerts@demo.org"])

### Самостійне завдання 4.2. Список без збігів

Застосуйте `re.findall` до тексту без електронних адрес і перевірте, що результатом є порожній список. Збережіть його у `result`.

**Очікування:** `[]`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
import re
text = "Контактну адресу буде повідомлено пізніше."
result = None
# TODO: використайте шаблон email і re.findall

if result is None:
    print("Допишіть пошук")
else:
    print(result)
    print("Самоперевірка:", result == [])

## 5. Форматування номерів: `re.sub` і групи

Шаблон `(\d{3})-(\d{3})-(\d{4})` має три групи: три, три й чотири цифри. У рядку заміни `r'(\1) \2-\3'` зворотні посилання вставляють відповідні групи. `re.sub` замінює всі збіги. Навчальний шаблон перевіряє форму, але не чинність номера; за потреби додаємо межі, щоб не захопити підрядок усередині довшої послідовності цифр.

In [ ]:
import re
contacts = "Служба: 044-123-4567; підтримка: 050-987-6543"
pattern = r"(?<!\d)(\d{3})-(\d{3})-(\d{4})(?!\d)"
replacement = r"(\1) \2-\3"
formatted = re.sub(pattern, replacement, contacts)
print(formatted)
assert formatted == "Служба: (044) 123-4567; підтримка: (050) 987-6543"

`(?<!\d)` і `(?!\d)` — перевірки сусідніх символів без включення їх до збігу: безпосередньо перед номером і після нього не повинно бути цифри. Заміну можна обмежити першим збігом через `count=1`. Початковий рядок не змінюється.

### Самостійне завдання 5.1. Телефонна книга

Змініть у багаторядковому тексті **всі** номери з `XXX-XXX-XXXX` на `(XXX) XXX-XXXX`. Збережіть текст у `result`.

**Очікування:** Обидва номери у новому форматі, імена без змін.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
import re
contacts = """Олена: 067-321-4567
Ігор: 093-111-2222"""
result = None
# TODO: pattern = ...; replacement = ...; result = re.sub(...)

if result is None:
    print("Допишіть групи і заміну")
else:
    print(result)
    print("Самоперевірка:", "(067) 321-4567" in result and "(093) 111-2222" in result)

### Самостійне завдання 5.2. Одна заміна

Для того самого формату номера змініть тільки **перший** збіг. Застосуйте `count=1`, збережіть у `result`.

**Очікування:** Перший номер переформатований, другий зберігає дефіси.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
import re
text = "перший 050-123-4567; другий 063-222-3333"
result = None
# TODO: re.sub(..., count=1)

if result is None:
    print("Додайте count=1")
else:
    print(result)
    print("Самоперевірка:", "(050) 123-4567" in result and "063-222-3333" in result)

## 6. Розділення за кількома знаками: `re.split`

Метод `str.split` приймає один фіксований роздільник, а `re.split` — шаблон. У `r'[;,!:\s-]+'` квадратні дужки означають **будь-який один** із перелічених знаків, `\s` — пробільний символ, `+` об’єднує послідовність таких знаків в один роздільник. Навіть з `+` можливі порожні елементи **на початку чи в кінці**, якщо текст там починається або закінчується роздільником. Тому фільтруємо всі порожні елементи, а не відрізаємо завжди останній.

In [ ]:
import re
text = "  Python - надійна; гнучка, популярна: мова!  "
pattern = r"[;,!:\s-]+"
raw_parts = re.split(pattern, text)
words = [word for word in raw_parts if word]
print("До фільтрації:", raw_parts)
print("Слова:", words)
assert words == ["Python", "надійна", "гнучка", "популярна", "мова"]

Цей шаблон покриває лише перелічені знаки; крапку або знак питання треба додавати окремо. Розбиття за пунктуацією може бути непридатним для чисел `3.14` і складних назв; правила завжди залежать від даних.

### Самостійне завдання 6.1. Список фруктів

Розділіть рядок за `,`, `;`, `!` і будь-якими пробільними символами. Приберіть порожні елементи і запишіть список у `result`.

**Очікування:** `['яблуко', 'груша', 'банан', 'ківі']`.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
import re
text = "  яблуко, груша;банан ! ківі  "
result = None
# TODO: створіть клас роздільників із +, застосуйте re.split і відфільтруйте порожні рядки

if result is None:
    print("Допишіть re.split і очищення результату")
else:
    print(result)
    print("Самоперевірка:", result == ["яблуко", "груша", "банан", "ківі"])

### Самостійне завдання 6.2. Коди з різними роздільниками

Розділіть список кодів за `|`, `/`, `;` та пробілами. Не розбивайте самі коди з дефісом.

**Очікування:** Чотири цілі коди, кожен із дефісом.  
**Як працювати:** допишіть код у наступній комірці на місці `# TODO`; виконайте її і порівняйте вивід з очікуванням.

In [ ]:
import re
text = "AB-1 | CD-2 / EF-3; GH-4"
result = None
# TODO: клас символів для чотирьох видів роздільників, але без дефіса

if result is None:
    print("Створіть шаблон без дефіса")
else:
    print(result)
    print("Самоперевірка:", result == ["AB-1", "CD-2", "EF-3", "GH-4"])

## 7. Підсумкове самостійне завдання

Є короткий текст повідомлення з номером, email і тегами. Напишіть програму, яка: (1) прибирає пробіли по краях; (2) витягає email через `re.findall`; (3) переформатовує номер через `re.sub`; (4) після `tags=` виділяє теги через `split(',')` та `strip()`; (5) створює словник `result` із ключами `email`, `formatted_text`, `tags`.

**Очікується:** `email` — список з однією адресою, `tags` — `['python', 'regex', 'practice']`, у `formatted_text` номер записано як `(067) 123-4567`. Працюйте в наступній комірці. Для відсутності email можна зберегти порожній список.

In [ ]:
import re
message = "  Контакт: student.lab@course.edu; телефон 067-123-4567; tags=python, regex, practice  "
result = None
# TODO 1: clean = message.strip()
# TODO 2: emails = re.findall(навчальний шаблон, clean)
# TODO 3: formatted_text = re.sub(шаблон із трьома групами, рядок заміни, clean)
# TODO 4: tags_text = ... (частина після 'tags='); tags = [...]
# TODO 5: result = {'email': emails, 'formatted_text': formatted_text, 'tags': tags}

if result is None:
    print("Заповніть п’ять кроків підсумкового завдання")
else:
    print(result)
    print("Самоперевірка:",
          result.get("email") == ["student.lab@course.edu"]
          and result.get("tags") == ["python", "regex", "practice"]
          and "(067) 123-4567" in result.get("formatted_text", ""))

## Контрольні запитання й висновки

1. Чому в `split('=', 1)` використано обмеження `1`? Що станеться, якщо значення містить `=`?
2. У чому різниця між `strip` і `translate`? Чи змінюється початковий рядок?
3. Чому `input()` перед форматуванням числа потребує перевірки й `int()`?
4. Чим `re.findall` відрізняється від `re.sub` і `re.split`?
5. Навіщо в шаблоні email записано `\.` замість `.`?
6. Чому список після `re.split` іноді містить порожні рядки?

**Підсумок:** тепер ви вмієте перетворювати рядок параметрів на словник, очищати та трансформувати текст, будувати числовий звіт, знаходити шаблони й змінювати формат даних. Збережіть власну копію через **Файл → Зберегти копію на Диску** або завантажте `.ipynb` після виконання вправ.